In [144]:
# --- Import Libraries ---
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import os

# --- Read Data ---
dfNightingale = pd.read_csv(
    'C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Data Visualization and Storytelling\\Assignment2\\data\\gabby_nightingale.csv'
)

# --- Data Preparation ---
dfNightingale['Date'] = pd.to_datetime(dfNightingale['Date'])
df = dfNightingale[dfNightingale['Date'] < '1855-04-01'].copy()

# Sort data by Year and Month
months_order = ['Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar']
df['Month'] = pd.Categorical(df['Month'].astype(str), categories=months_order, ordered=True)
df = df.sort_values(['Year', 'Month']).reset_index(drop=True)

# --- Setup variables ---
yearsFromDf = df['Year'].unique().tolist()
deathTypes = ['Disease', 'Wounds', 'Other']
colours = {'Disease': 'red', 'Wounds': 'lightcoral', 'Other': 'pink'}
maxValue = df[deathTypes].max().max()

# --- Create Coxcomb Chart ---
fig = go.Figure()
for year in yearsFromDf:
    dfYear = df[df['Year'] == year]
    for death_type in deathTypes:
        fig.add_trace(go.Barpolar(
            r=dfYear[death_type],
            theta=dfYear['Month'],
            name=f'{death_type} ({year})',
            marker_color=colours[death_type],
                    hovertemplate=f"<b>{death_type} Deaths</b><br>" +
                    "Month: %{theta}<br>" +
                    "Deaths: %{r}<br>" +
                    "<i>Click for line image</i><extra></extra>",
            visible=True if year == yearsFromDf[0] else False
        ))

# --- Function to create visibility ---
def createVisibility(selected_year, selected_death_type):
    visibility = []
    for year in yearsFromDf:
        for death_type in deathTypes:
            if selected_year == "All Years" and selected_death_type == "All Deaths":
                visibility.append(True)
            elif selected_year == "All Years":
                visibility.append(death_type == selected_death_type)
            elif selected_death_type == "All Deaths":
                visibility.append(year == int(selected_year))
            else:
                visibility.append(year == int(selected_year) and death_type == selected_death_type)
    return visibility

# Create dropdown buttons with dynamic title updates: This is not working on the Flask server but works in Jupyter Notebook
yearDropdown = []
for y in ["All Years"] + [str(y) for y in yearsFromDf]:
    args = [{"visible": createVisibility(y, "All Deaths")}, 
            {"title": f"<b>Redesign of Florence Nightingale's Coxcomb Chart (Before Reform)- {y}, All Deaths</b>"}]
    yearDropdown.append({"label": y, "method": "update", "args": args})

deathDropdown = []
for dt in ["All Deaths"] + deathTypes:
    args = [{"visible": createVisibility(str(yearsFromDf[0]), dt)}, 
            {"title": f"<b>Redesign of Florence Nightingale's Coxcomb Chart (Before Reform) - {yearsFromDf[0]}, {dt}</b>"}]
    deathDropdown.append({"label": dt, "method": "update", "args": args})

# --- Layout ---
fig.update_layout(
    updatemenus=[
        {"buttons": yearDropdown, "direction": "down", "showactive": True,
         "x": 0.80, "xanchor": "left", "y": 1.02, "yanchor": "top"},
        {"buttons": deathDropdown, "direction": "down", "showactive": True,
         "x": 0.80, "xanchor": "left", "y": 0.85, "yanchor": "top"}
    ],
    title={
        'text': f"<b>Redesign of Florence Nightingale's Coxcomb Chart (Before Reform)- All Years, All Deaths</b>",
        'font': {'size': 32, 'color': 'black', 'family': 'Arial'},
        'x': 0.5,
        'xanchor': 'center'
    },
    template=None,
    polar=dict(
        bgcolor="rgb(255,240,249)",
        radialaxis=dict(
            range=[0, maxValue],
            showticklabels=False,
            showline=True,
            linewidth=1,
            linecolor="white",
            gridcolor="white",
            ticks=''
        ),
        angularaxis=dict(
            direction='clockwise',
            linewidth=1,
            showline=False,
            linecolor='red',
            gridcolor='lightgrey',
            tickfont=dict(size=11, color='darkgrey')
        )
    ),
    legend=dict(title={"text": "Death Types"}, x=0.85, y=0.15, xanchor="left", yanchor="bottom"),
    margin=dict(r=200)
)

fig.show()

# --- Save HTML with interactive JS ---
html_str = fig.to_html(include_plotlyjs='cdn', full_html=True)

interactive_script = f"""
<script>
const MAX_X = {maxValue};

function attachClickHandler() {{
    const plot = document.getElementsByClassName('plotly-graph-div')[0];
    if (!plot) {{ setTimeout(attachClickHandler, 100); return; }}

    plot.on('plotly_click', function(data) {{
        const point = data.points[0];
        const month = point.theta.trim();
        const year = (point.data.name.match(/\\d{{4}}/) || ['Unknown'])[0];
        const deathType = point.data.name.split(' ')[0];

        fetch(`/get_dots/${{deathType}}/${{month}}/${{year}}`)
        .then(resp => resp.json())
        .then(dots => renderDotsLine(deathType, month, year, dots))
        .catch(err => console.error(err));
    }});
}}

function renderDotsLine(deathType, month, year, dots) {{
    let div = document.getElementById('dots-chart');
    if (!div) {{
        div = document.createElement('div');
        div.id = 'dots-chart';
        div.style.marginTop = '40px';
        document.body.appendChild(div);
    }}

    if (!dots.length) {{
        Plotly.purge('dots-chart');
        div.innerHTML = '<p>No data available.</p>';
        return;
    }}

    const trace = {{
        x: dots.map((d,i) => i+1),
        y: dots.map(() => 0),
        mode: 'markers',
        marker: {{color: 'red', size: 10, symbol: 'circle'}},
        type: 'scatter'
    }};

    const layout = {{
        title: `${{deathType}} Deaths in ${{month}} ${{year}}`,
        xaxis: {{title: 'Deaths', range: [0, MAX_X], showgrid: true, zeroline: false}},
        yaxis: {{visible: false}},
        margin: {{t:50, l:40, r:30, b:40}},
        height: 250,
        plot_bgcolor: 'white',
        paper_bgcolor: 'white'
    }};

    Plotly.newPlot('dots-chart', [trace], layout);
}}

document.addEventListener("DOMContentLoaded", attachClickHandler);
</script>
"""

final_html = html_str.replace("</body>", f"{interactive_script}</body>")

output_path = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Data Visualization and Storytelling\\Assignment2\\templates\\nightingale_chart.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_html)

print(f"✅ HTML with interactivity saved to: {output_path}")


✅ HTML with interactivity saved to: C:\Users\afeen\Documents\MBAI\Business Analytics\FlaskProjects\Data Visualization and Storytelling\Assignment2\templates\nightingale_chart.html


In [145]:
# --- Import Libraries ---
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import os

# --- Read Data ---
dfNightingale2 = pd.read_csv(
    'C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Data Visualization and Storytelling\\Assignment2\\data\\gabby_nightingale.csv'
)
dfNightingale2['Date'] = pd.to_datetime(dfNightingale2['Date'])
dfNightingale2 = dfNightingale2[dfNightingale2['Date'] >= '1854-04-01'].copy()

# --- Define death types ---
deathTypes = ['Disease', 'Wounds', 'Other']

# --- Prepare data for line graph ---
dfNightingale2['Month_Year'] = dfNightingale2['Date'].dt.to_period('M').dt.to_timestamp()

# Unpivot dataframe to long format
df_line = dfNightingale2.melt(
    id_vars=['Month_Year'],
    value_vars=deathTypes,
    var_name='Death Type',
    value_name='Deaths'
)

# --- Create px.line figure ---
fig_line = px.line(
    df_line,
    x='Month_Year',
    y='Deaths',
    color='Death Type',
    title='<b>Impact of Nightingale\'s Sanitary Reforms on Death Counts</b>', 
    labels={'Month_Year': 'Month and Year', 'Deaths': 'Number of Deaths'},
    line_shape='linear'
)

# --- Convert to go.Figure to safely add vline ---
fig_line = go.Figure(fig_line)

# Identify reform date
reform_start = pd.Timestamp('1855-01-01')
fig_line.add_shape(
    type="line",
    x0=reform_start,
    y0=0,
    x1=reform_start,
    y1=df_line['Deaths'].max(),
    line=dict(color="lightgrey", width=1, dash="dot"),
)
fig_line.add_annotation(
    x=reform_start,
    y=df_line['Deaths'].max(),
    text="<i><b>Reform Start</b></i>",
    font=dict(color="grey", size=12),
    showarrow=True,
    arrowhead=1,
    ax=0,
    ay=-40
)

# --- Customize line colors ---
fig_line.update_layout(template=None) # remove background colour
fig_line.update_yaxes(showline=True, linewidth=1, linecolor='lightgrey', mirror=False, showgrid=False)
fig_line.update_xaxes(showgrid=False) # remove y-axis gridlines
fig_line.update_traces(selector=dict(name='Disease'), line=dict(color='red'))
fig_line.update_traces(selector=dict(name='Wounds'), line=dict(color='lightcoral'))
fig_line.update_traces(selector=dict(name='Other'), line=dict(color='pink'))

fig_line.show()

# make file write to Templates folder for Flask

final_html = fig_line.to_html(full_html=True, include_plotlyjs='cdn')

output_path = "C:\\Users\\afeen\\Documents\\MBAI\\Business Analytics\\FlaskProjects\\Data Visualization and Storytelling\\Assignment2\\templates\\nightingale_impact_after_reform.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_html)

print(f"✅ Line chart saved to {output_path}")


✅ Line chart saved to C:\Users\afeen\Documents\MBAI\Business Analytics\FlaskProjects\Data Visualization and Storytelling\Assignment2\templates\nightingale_impact_after_reform.html
